# Parametric PCA-MNIST exact-polar Muon experiment

This notebook has no dependency on the original large `mnist_muon.zip`.

- `PROBLEM_MODE = "frozen"` uses the included exact 2.1 MB problem.
- `PROBLEM_MODE = "rebuild"` refits PCA from the included 9.4 MB uint8 MNIST training set.

The main parameter cell exposes `n_train`, PCA dimension and whitening, teacher rank, student width, fixed learning rate, steps, seed lists, and analysis windows.


In [ ]:
from pathlib import Path
import copy
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "mnist_pca_muon.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import mnist_pca_muon as mpm
import pandas as pd
from IPython.display import Image, display

print("Bundle root:", ROOT)

## 1. Parameters

In [ ]:
# ------------------------ EDIT THIS CELL ------------------------
# "frozen": exact reported problem; no raw image file is read.
# "rebuild": rebuild PCA/teacher from data/mnist_train_uint8.npz (9.4 MB).
PROBLEM_MODE = "frozen"       # "frozen" or "rebuild"

# Use reproduction settings by default. This avoids accidentally running the
# tiny smoke test and comparing its numbers with the reported experiment.
PRESET = "reproduction"       # "reproduction" or "smoke"
RUN_CONFIRMATION = False      # set True for all 30 confirmation seeds
FORCE_REBUILD_PROBLEM = True  # relevant only in rebuild mode
FORCE_RERUN = True

FROZEN_PROBLEM = ROOT / "data" / "fixed_problem_reproduction.npz"
LIGHTWEIGHT_MNIST = ROOT / "data" / "mnist_train_uint8.npz"

config = mpm.load_config(ROOT / "configs" / f"{PRESET}.json")
config["output"]["output_root"] = str(ROOT / "notebook_outputs")
config["output"]["experiment_name"] = f"notebook_{PRESET}_{PROBLEM_MODE}"

# Main editable parameters.
if PRESET == "reproduction":
    config["data"]["n_train"] = 5000
    config["data"]["pca_dim"] = 100
    config["data"]["whiten"] = True
    config["data"]["problem_seed"] = 2026
    config["data"]["allowed_digits"] = None
    config["teacher"]["rank"] = 4
    config["teacher"]["seed"] = 5
    config["teacher"]["head_value"] = 1.0
    config["student"]["width"] = 50
    config["student"]["head_value"] = None
    config["student"]["learning_rate"] = 0.60
    config["student"]["steps"] = 1500
    config["student"]["log_every"] = 10
    config["student"]["log_both_phases"] = True
    config["student"]["threads"] = 1
    config["analysis"]["primary_window"] = [60, 240]
else:
    config["data"]["n_train"] = 600
    config["data"]["pca_dim"] = 20
    config["teacher"]["rank"] = 2
    config["student"]["width"] = 12
    config["student"]["learning_rate"] = 0.40
    config["student"]["steps"] = 50
    config["student"]["log_every"] = 5
    config["analysis"]["primary_window"] = [10, 30]

# Rebuild mode reads the compact direct NPZ, not the original large zip.
config["data"]["mnist_source"] = str(LIGHTWEIGHT_MNIST)
config["data"]["images_key"] = "images"
config["data"]["labels_key"] = "labels"
config["data"]["pixel_scale"] = 255.0

# Examples of modifications:
# config["data"]["n_train"] = 10000
# config["data"]["pca_dim"] = 150
# config["data"]["allowed_digits"] = [0, 1, 2, 3]
# config["teacher"]["rank"] = 6
# config["student"]["width"] = 80
# config["student"]["learning_rate"] = 0.50
# config["student"]["steps"] = 2500
# config["analysis"]["primary_window"] = [100, 500]

mpm.validate_config(config)
print("Preset:", PRESET)
print("Problem mode:", PROBLEM_MODE)
print("Frozen problem exists:", FROZEN_PROBLEM.exists())
print("Lightweight MNIST exists:", LIGHTWEIGHT_MNIST.exists())
print("Confirmation seeds:", len(config["seeds"]["confirmation"]))
config


## 2. Prepare the fixed problem

In [ ]:
import shutil

experiment_dir = mpm.experiment_dir(config)
experiment_dir.mkdir(parents=True, exist_ok=True)
problem_path = experiment_dir / config["output"]["problem_filename"]

if PROBLEM_MODE == "frozen":
    if PRESET != "reproduction":
        raise ValueError(
            "The included frozen problem matches the reproduction preset. "
            "Use PRESET='reproduction' or switch to PROBLEM_MODE='rebuild'."
        )
    if not FROZEN_PROBLEM.is_file():
        raise FileNotFoundError(FROZEN_PROBLEM)
    shutil.copy2(FROZEN_PROBLEM, problem_path)
else:
    if not LIGHTWEIGHT_MNIST.is_file():
        raise FileNotFoundError(
            f"Compact MNIST file is missing: {LIGHTWEIGHT_MNIST}"
        )
    problem_path = mpm.prepare_problem(
        config,
        force=FORCE_REBUILD_PROBLEM,
    )

problem = mpm.load_problem(problem_path)
print("Problem:", problem_path)
print("X:", problem["X"].shape)
print("Teacher basis:", problem["U_teacher"].shape)

if PRESET == "reproduction":
    assert problem["X"].shape == (5000, 100)
    assert problem["U_teacher"].shape == (100, 4)

summary_path = problem_path.with_name("problem_summary.csv")
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    display(pd.DataFrame([{
        "n_train": len(problem["X"]),
        "pca_dimension": problem["X"].shape[1],
        "teacher_rank": problem["U_teacher"].shape[1],
        "source": "included frozen problem",
    }]))


## 3. Train students

The full-batch update is $W_{t+1}=W_t-\eta\,\mathrm{Polar}(\nabla_W L(W_t))$. The head is fixed.

In [ ]:
groups = ("development", "confirmation") if RUN_CONFIRMATION else ("development",)
run_outputs = mpm.run_experiment(
    config,
    groups=groups,
    rebuild_problem=False,
    force_runs=FORCE_RERUN,
)
run_outputs

## 4. Analyze the loss cycle and both alignment phases

In [ ]:
analysis_outputs = mpm.analyze_experiment(config, groups=groups)
summary = pd.read_csv(analysis_outputs["summary"])
summary

In [ ]:
analysis_dir = analysis_outputs["analysis_dir"]
for group in groups:
    for filename in [
        f"{group}_loss_cycle.png",
        f"{group}_mean_alignment_two_phases.png",
        f"{group}_min_alignment_two_phases.png",
        f"{group}_mean_alignment_phase_average.png",
        f"{group}_min_alignment_phase_average.png",
        f"{group}_two_step_residual.png",
    ]:
        path = analysis_dir / filename
        if path.exists():
            display(Image(filename=str(path)))

## 5. Optional fixed-learning-rate sweep

In [ ]:
RUN_ETA_SWEEP = False
ETA_GRID = [0.4, 0.5, 0.6, 0.7, 0.8]

if RUN_ETA_SWEEP:
    sweep_config = copy.deepcopy(config)
    sweep_config["output"]["experiment_name"] = f"notebook_{PRESET}_sweep"
    sweep_table = mpm.sweep_learning_rates(
        sweep_config,
        ETA_GRID,
        group="development",
        force=FORCE_RERUN,
    )
    display(sweep_table)
else:
    print("Set RUN_ETA_SWEEP=True to run the sweep.")

The teacher basis is drawn independently in PCA coordinates. Whitening controls covariance but does not make the empirical MNIST distribution Gaussian. The task still uses teacher-generated targets.